In [1]:
import numpy as np
from tabulate import tabulate
from magres.atoms import MagresAtoms
atoms = MagresAtoms.load_magres('/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/CASTEP/8-HQ-ipc2-B_opt_magres_new.magres')

In [2]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [3]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [4]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g
    

for atom in atoms.species('B'):
    print (atom, "sigma:\n",atom.ms.sigma)
    print()

In [5]:
for atom in atoms.species('B'):
    print (atom, "sigma:\n",atom.efg.Cq)
    print()

11B1 sigma:
 -2.116431749994685

11B2 sigma:
 -2.1164317499945637

11B3 sigma:
 -2.1164317499947103

11B4 sigma:
 -2.116431749994567



In [16]:
for atom in atoms.species('B'):
    print (atom, "sigma:\n",atom.ms.sigma)
    print()

11B1 sigma:
 [[90.56439481  4.06748529  1.69557227]
 [-2.08205741 84.48173583  6.71136584]
 [-4.05926981  2.21640125 76.98631145]]

11B2 sigma:
 [[90.56439481 -4.06748529  1.69557227]
 [ 2.08205741 84.48173583 -6.71136584]
 [-4.05926981 -2.21640125 76.98631145]]

11B3 sigma:
 [[90.56439481  4.06748529  1.69557227]
 [-2.08205741 84.48173583  6.71136584]
 [-4.05926981  2.21640125 76.98631145]]

11B4 sigma:
 [[90.56439481 -4.06748529  1.69557227]
 [ 2.08205741 84.48173583 -6.71136584]
 [-4.05926981 -2.21640125 76.98631145]]



In [ ]:
efg_tensor = atoms.species('B')[0].efg.V  # Extract the EFG tensor as a numpy array
print(efg_tensor)

# EFG tensor from magres
# efg[0,0]= -0.0286; efg[0,1]= 0.1873 ; efg[0,2]= 0.1114;
# efg[1,0]= efg[0,1]; efg[1,1]= -0.0059; efg[1,2]= 0.0265;
# efg[2,0]= efg[0,2]; efg[2,1]= efg[1,2]; efg[2,2]= 0.0345;

[[-0.02858682  0.18732633  0.11138392]
 [ 0.18732633 -0.00589699  0.02648839]
 [ 0.11138392  0.02648839  0.03448382]]


In [24]:
# using values from latest magres file
Cs = np.zeros((3, 3))                                # CS symmetric (l = 0 + 2) Tensor from updated_magres
CS_anti = np.zeros((3,3))                           # CS antisymmetric ( l = 1)
CS_iso = np.zeros((3,3))                            # CS isotropic  (l = 0)
CS_total = np.zeros((3,3))                            # CS total shielding tensor ( l = 0 + 1 + 2) Tensor from magres

CS_total[:,:] = atoms.species('B').ms.sigma[0]

iso = np.mean([CS_total[0,0], CS_total[1,1], CS_total[2,2]]) # isotropic chemical shielding (l = 0)

CS_iso[0,0] = CS_iso[1,1] = CS_iso[2,2] = iso

Cs[0,0] = CS_total[0,0]; Cs[0,1] = (CS_total[0,1] + CS_total[1,0] )/2; Cs[0,2] = (CS_total[0,2] + CS_total[2,0])/2;
Cs[1,0] = Cs[0,1];      Cs[1,1] = CS_total[1,1];                      Cs[1,2] = (CS_total[1,2] + CS_total[2,1])/2;
Cs[2,0] = Cs[0,2];      Cs[2,1] = Cs[1,2];                           Cs[2,2] = CS_total[2,2];

CS_anti[0,1] = (CS_total[0,1] - CS_total[1,0])/2; CS_anti[0,2] = (CS_total[0,2] - CS_total[2,0])/2; 
CS_anti[1,0] = -CS_anti[0,1]; CS_anti[1,2] = (CS_total[1,2] - CS_total[2,1])/2;
CS_anti[2,0] = -CS_anti[0,2];      CS_anti[2,1] = -CS_anti[1,2]; 


efg = np.zeros((3, 3 ))                             # EFG Tensor from magres (in a.u.)

efg[:,:] = atoms.species('B')[0].efg.V


# Convert a.u. units to MHz
# Q tensor elements (MHz) = efg tensor (a.u.)* Q (barn) * 234.9647 
# Q = 0.04059 barn https://www-nds.iaea.org/publications/indc/indc-nds-0650.pdf
Q = 0.04059
V = efg*Q*234.9647

print('\nQ tensor:\n', np.round(V,3))
print('\nCS Tensor:\n',np.round(CS_total, 3))
print('\nCS isotropic Tensor:\n',np.round(CS_iso, 3))
print('\nCS symmetric Tensor:\n',np.round(Cs,3))
print('\nCS antisymmetric Tensor:\n',np.round(CS_anti,3))



Q tensor:
 [[-0.273  1.787  1.062]
 [ 1.787 -0.056  0.253]
 [ 1.062  0.253  0.329]]

CS Tensor:
 [[90.564  4.067  1.696]
 [-2.082 84.482  6.711]
 [-4.059  2.216 76.986]]

CS isotropic Tensor:
 [[84.011  0.     0.   ]
 [ 0.    84.011  0.   ]
 [ 0.     0.    84.011]]

CS symmetric Tensor:
 [[90.564  0.993 -1.182]
 [ 0.993 84.482  4.464]
 [-1.182  4.464 76.986]]

CS antisymmetric Tensor:
 [[ 0.     3.075  2.877]
 [-3.075  0.     2.247]
 [-2.877 -2.247  0.   ]]


In [21]:
print(np.mean([CS_total[0,0], CS_total[1,1]]))

87.52306532117157


In [8]:
sorted_eigenvalues_efg, dc_efg, quad_avg, eigenvalues_efg, eigenvectors_efg = sort_eigenvalues(V)
print('==================================\n')
sorted_eigenvalues_cs, dc_cs, cs_avg, eigenvalues_cs, eigenvectors_cs = sort_eigenvalues(Cs)

 Unsorted Eigenvalues:
 [ 2.11526380e+00 -2.11629537e+00  1.03156610e-03] 

 Unsorted Eigenvectors:
 [[-0.65300979 -0.74505746 -0.13589549]
 [-0.59212743  0.61413793 -0.52174679]
 [-0.47218992  0.26023832  0.84220704]] 

Sorted Eigenvalues: 
 [ 1.03156610e-03  2.11526380e+00 -2.11629537e+00] 

Sorted Eigenvectors: 
 [[-0.13589549 -0.65300979 -0.74505746]
 [-0.52174679 -0.59212743  0.61413793]
 [ 0.84220704 -0.47218992  0.26023832]] 


 Unsorted Eigenvalues:
 [74.7647565  90.74304439 86.5246412 ] 

 Unsorted Eigenvectors:
 [[-0.09400673  0.99110348 -0.09421587]
 [ 0.42351385  0.12545603  0.89716041]
 [-0.90099875 -0.04443739  0.43153976]] 

Sorted Eigenvalues: 
 [86.5246412  90.74304439 74.7647565 ] 

Sorted Eigenvectors: 
 [[-0.09421587  0.99110348 -0.09400673]
 [ 0.89716041  0.12545603  0.42351385]
 [ 0.43153976 -0.04443739 -0.90099875]] 



In [9]:

#Calculate Quadrupolar Tensor in PAS

Vyy = sorted_eigenvalues_efg[0]
Vxx = sorted_eigenvalues_efg[1]
Vzz = sorted_eigenvalues_efg[2]

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================')

#Calculate CSA Tensor in PAS

Csyy = sorted_eigenvalues_cs[0] 
Csxx = sorted_eigenvalues_cs[1]  
Cszz = sorted_eigenvalues_cs[2]



print('CSA Tensor Components δyy, δxx, δzz: \n', Csyy, Csxx, Cszz)

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 0.0010315661027169962 2.115263801471579 -2.1162953675742955
CSA Tensor Components δyy, δxx, δzz: 
 86.5246411957876 90.74304439418816 74.76475650335016


In [10]:
iso_cs = (Csxx + Csyy + Cszz)/3

csa = Cszz - iso_cs
etas = (Csyy - Csxx)/csa

#for Quadrupolar
CQ_fit = Vzz

etaq = (Vyy - Vxx)/Vzz

table = [['CQ (MHz)', CQ_fit], ['etaq', etaq ], ['iso_cs (ppm)',iso_cs ],['csa (ppm)', csa],  ['etas', etas]  ]
print(tabulate(table, headers=['Qauntity', 'Value']))

Qauntity          Value
------------  ---------
CQ (MHz)      -2.1163
etaq           0.999025
iso_cs (ppm)  84.0108
csa (ppm)     -9.24606
etas           0.456238


In [11]:
# Calculation for efg tensor
print('Direction cosine efg:\n')
print(dc_efg, '\n')
a_efg, b_efg, g_efg = get_euler_angles(dc_efg)

print("Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:")
print(a_efg, b_efg, g_efg, '\n')

print('=========================')
print('Direction cosine csa: \n')
print(dc_cs, '\n')
a_cs, b_cs, g_cs = get_euler_angles(dc_cs)

print("Calculated Euler angles (degrees) CSA PAS --> Crystal:")
print(a_cs, b_cs, g_cs, '\n')

Direction cosine efg:

[[-0.65300979 -0.13589549 -0.74505746]
 [-0.59212743 -0.52174679  0.61413793]
 [-0.47218992  0.84220704  0.26023832]] 

Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:
-60.72249166509776 74.91579652304037 39.49816625591709 

Direction cosine csa: 

[[ 0.99110348 -0.09421587 -0.09400673]
 [ 0.12545603  0.89716041  0.42351385]
 [-0.04443739  0.43153976 -0.90099875]] 

Calculated Euler angles (degrees) CSA PAS --> Crystal:
-84.12074423568313 154.2896611261801 77.48504630443955 



In [12]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(dc_efg), (dc_cs))

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: -47.46453173518223 chi: 84.51058053947129 xi: 76.28200639420761 



**Rotation of tensors Crystal--> Tenon Frame**

In [13]:
#Euler angles Crystal--> Tenon Frame for LHQ
alpha = 280
beta = 72.5
gamma = 180
U  = Rabc(alpha, beta, gamma)
Cs_tenon = np.matmul(np.matmul(np.linalg.inv(U), Cs), U)

V_tenon = np.matmul(np.matmul(np.linalg.inv(U), V), U)

print('CSA Tensor in Crystal Frame: \n', Cs) #from magres
print('CSA Tensor in Tenon Frame: \n', Cs_tenon)

print('==========================')
print('Quad Tensor in Crystal Frame: \n', V)  #from magres
print('Quad Tensor in Tenon Frame: \n', V_tenon)


CSA Tensor in Crystal Frame: 
 [[90.56439481  0.99271394 -1.18184877]
 [ 0.99271394 84.48173583  4.46388355]
 [-1.18184877  4.46388355 76.98631145]]
CSA Tensor in Tenon Frame: 
 [[82.95920605  4.6759289  -3.09861062]
 [ 4.6759289  80.41450615  4.39078599]
 [-3.09861062  4.39078599 88.6587299 ]]
Quad Tensor in Crystal Frame: 
 [[-0.27276441  1.78632078  1.06244599]
 [ 1.78632078 -0.05626958  0.25273626]
 [ 1.06244599  0.25273626  0.32903399]]
Quad Tensor in Tenon Frame: 
 [[ 0.03661077 -0.23063248 -1.57151032]
 [-0.23063248 -0.42765735 -1.33608148]
 [-1.57151032 -1.33608148  0.39104658]]
